# 03 BM25 Retrieval PoC (GermanQuAD + GerLeRB)

Dieses Notebook implementiert einen reproduzierbaren BM25-Retrieval-PoC auf den **normalisierten** Datensätzen in `code/output/<dataset>/`.

Fokus:
- Retrieval-only (kein Query-Rewriting, keine Generierung, kein Hybrid Retrieval)
- identische Retrieval- und Evaluationslogik für beide Datensätze
- strukturierter Ergebnisexport nach `code/output/bm25/`

## Methodik und Metriken

**Eingaben pro Datensatz**
- `queries.normalized.jsonl`
- `docs.normalized.jsonl`
- `qrels.normalized.jsonl`

**Schema (normalisiert)**
- Query: `query_id`, `query_text`
- Dokument: `doc_id`, `text`
- Qrels: `query_id`, `doc_id`, `relevance`

**Preprocessing / Tokenisierung**
- einfache, deterministische deutsche Tokenisierung per Regex
- Lowercasing
- keine externen NLP-Modelle
- Dokumenttext: `text`

**BM25**
- Retriever: `rank_bm25.BM25Okapi`
- Paketversion wird beim Lauf mit `importlib.metadata` protokolliert
- Parameter: `k1`, `b`, `epsilon`

**k-Werte und Relevanz**
- `k_values = [1, 5, 10]`
- Relevanzschwelle: `relevance >= 1`

**Seed / Determinismus**
- BM25 selbst verwendet keinen Zufallsseed
- Tie-Break bei gleichen Scores deterministisch über `doc_id`

**Metrikdefinitionen**

Sei $Q$ die Menge aller Queries im Datensatz, $G_q$ die Menge der bekannten relevanten Dokumente für Query $q$ und $T_{q,k}$ die Top-$k$-Menge.

Recall@k:
$$
Recall@k = \\frac{1}{|Q|} \\sum_{q \\in Q} \\frac{|G_q \\cap T_{q,k}|}{|G_q|}
$$
Queries ohne bekannte relevante Dokumente werden mit 0 berichtet und separat gezählt.

Success@k / Hit Rate@k:
$$
Success@k = \\frac{1}{|Q|} \\sum_{q \\in Q} \\mathbb{1}\\left(|G_q \\cap T_{q,k}| > 0\\right)
$$

MRR@k:
$$
MRR@k = \\frac{1}{|Q|} \\sum_{q \\in Q} \\begin{cases}
\\frac{1}{rank_q}, & \\text{falls ein relevantes Dokument in Top}_k\\text{ gefunden wird}\\\\
0, & \\text{sonst}
\\end{cases}
$$
wobei $rank_q$ die 1-basierte Position des ersten relevanten Dokuments in Top-$k$ ist.

Bei den aktuell lokal vorliegenden qrels gibt es je Query genau ein bekanntes relevantes Dokument. Daher sind Recall@k und Success@k numerisch gleich; sie werden dennoch getrennt ausgewiesen, damit die Kennzahlen bei künftigen Multi-Relevanz-qrels korrekt interpretierbar bleiben.

In [1]:
from __future__ import annotations

import json
import platform
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from importlib.metadata import version as package_version
from pathlib import Path
from typing import Any

import pandas as pd
import sys

# Reproduzierbare Basiskonfiguration
K_VALUES = [1, 5, 10]
RELEVANCE_THRESHOLD = 1
BM25_K1 = 1.5
BM25_B = 0.75
BM25_EPSILON = 0.25
SAMPLE_QUERY_COUNT = 5
PYTHON_VERSION = platform.python_version()
RANK_BM25_VERSION = package_version("rank-bm25")

# Notebook kann aus Repo-Root oder aus code/ gestartet werden
ROOT = Path.cwd().resolve()
if not (ROOT / "code").exists() and ROOT.name == "code":
    ROOT = ROOT.parent

CODE_DIR = ROOT / "code"
OUTPUT_DIR = CODE_DIR / "output"
BM25_OUTPUT_DIR = OUTPUT_DIR / "bm25"
PLOT_DIR = CODE_DIR / "plot"
PYLIB_DIR = CODE_DIR / "pylib"

BM25_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
if str(PYLIB_DIR) not in sys.path:
    sys.path.insert(0, str(PYLIB_DIR))

DATASETS = {
    "germanquad": {
        "queries": OUTPUT_DIR / "germanquad" / "queries.normalized.jsonl",
        "docs": OUTPUT_DIR / "germanquad" / "docs.normalized.jsonl",
        "qrels": OUTPUT_DIR / "germanquad" / "qrels.normalized.jsonl",
    },
    "gerlerb": {
        "queries": OUTPUT_DIR / "gerlerb" / "queries.normalized.jsonl",
        "docs": OUTPUT_DIR / "gerlerb" / "docs.normalized.jsonl",
        "qrels": OUTPUT_DIR / "gerlerb" / "qrels.normalized.jsonl",
    },
}

print(f"ROOT={ROOT}")
print(f"python={PYTHON_VERSION}")
print(f"rank-bm25={RANK_BM25_VERSION}")
print(f"Output wird geschrieben nach: {BM25_OUTPUT_DIR}")

ROOT=<PROJEKT>
python=3.9.6
rank-bm25=0.2.2
Output wird geschrieben nach: <PROJEKT>/code/output/bm25


In [2]:
try:
    from rank_bm25 import BM25Okapi
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Paket 'rank_bm25' ist nicht installiert. Bitte installiere es vor Ausführung \\nmit: pip install rank-bm25"
    ) from exc

from local_dataset_io import load_jsonl

In [3]:
TOKEN_PATTERN = re.compile(r"[0-9A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df]+")


def tokenize_de(text: str) -> list[str]:
    text = (text or "").lower()
    return TOKEN_PATTERN.findall(text)


def compose_doc_text(doc: dict[str, Any]) -> str:
    title = str(doc.get("title") or "").strip()
    text = str(doc.get("text") or "").strip()
    if title and text:
        return f"{title}\n{text}"
    return title or text


@dataclass
class DatasetBundle:
    name: str
    split_values: list[str]
    queries: list[dict[str, Any]]
    docs: list[dict[str, Any]]
    qrels: list[dict[str, Any]]


def load_dataset(name: str, paths: dict[str, Path]) -> DatasetBundle:
    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Fehlende normalisierte Datei(en) für Datensatz '"
            f"{name}': {missing}. Erwartet werden Dateien unter code/output/<dataset>/."
        )

    queries = load_jsonl(paths["queries"])
    docs = load_jsonl(paths["docs"])
    qrels = load_jsonl(paths["qrels"])

    required_query_keys = {"query_id", "query_text"}
    required_doc_keys = {"doc_id", "text"}
    required_qrel_keys = {"query_id", "doc_id", "relevance"}

    if queries and not required_query_keys.issubset(queries[0].keys()):
        raise ValueError(f"Ungültiges Query-Schema in {name}: {queries[0].keys()}")
    if docs and not required_doc_keys.issubset(docs[0].keys()):
        raise ValueError(f"Ungültiges Doc-Schema in {name}: {docs[0].keys()}")
    if qrels and not required_qrel_keys.issubset(qrels[0].keys()):
        raise ValueError(f"Ungültiges Qrels-Schema in {name}: {qrels[0].keys()}")

    split_values = sorted(
        {str(x.get("split", "unknown")) for x in (queries + docs + qrels)}
    )

    return DatasetBundle(
        name=name,
        split_values=split_values,
        queries=queries,
        docs=docs,
        qrels=qrels,
    )

In [4]:
def evaluate_dataset(bundle: DatasetBundle, k_values: list[int], relevance_threshold: int) -> dict[str, Any]:
    docs_by_id = {str(d["doc_id"]): d for d in bundle.docs}
    doc_ids = list(docs_by_id.keys())

    corpus_tokens = [tokenize_de(compose_doc_text(docs_by_id[doc_id])) for doc_id in doc_ids]
    bm25 = BM25Okapi(corpus_tokens, k1=BM25_K1, b=BM25_B, epsilon=BM25_EPSILON)

    qrels_by_query: dict[str, set[str]] = {}
    for row in bundle.qrels:
        qid = str(row["query_id"])
        did = str(row["doc_id"])
        rel = int(row["relevance"])
        if rel >= relevance_threshold and did in docs_by_id:
            qrels_by_query.setdefault(qid, set()).add(did)

    max_k = max(k_values)
    query_rows: list[dict[str, Any]] = []

    queries_without_qrels = 0
    queries_without_relevant_docs = 0

    for q in bundle.queries:
        qid = str(q["query_id"])
        qtext = str(q["query_text"])
        q_tokens = tokenize_de(qtext)

        scores = bm25.get_scores(q_tokens)
        ranked = sorted(
            zip(doc_ids, scores),
            key=lambda item: (-float(item[1]), item[0]),
        )
        top_ranked = ranked[:max_k]

        gold_docs = qrels_by_query.get(qid, set())
        has_qrels = any(str(r["query_id"]) == qid for r in bundle.qrels)
        if not has_qrels:
            queries_without_qrels += 1
        if not gold_docs:
            queries_without_relevant_docs += 1

        top_doc_ids = [doc_id for doc_id, _ in top_ranked]
        top_doc_scores = [float(score) for _, score in top_ranked]

        first_rel_rank = None
        for rank_idx, doc_id in enumerate(top_doc_ids, start=1):
            if doc_id in gold_docs:
                first_rel_rank = rank_idx
                break

        query_rows.append(
            {
                "dataset": bundle.name,
                "query_id": qid,
                "query_text": qtext,
                "gold_doc_ids": sorted(gold_docs),
                "top_doc_ids": top_doc_ids,
                "top_doc_scores": top_doc_scores,
                "first_rel_rank": first_rel_rank,
            }
        )

    query_count = len(query_rows)
    metrics: list[dict[str, Any]] = []
    for k in k_values:
        recall_sum = 0.0
        success_count = 0
        mrr_sum = 0.0
        for row in query_rows:
            gold_set = set(row["gold_doc_ids"])
            retrieved = row["top_doc_ids"][:k]
            hit_count = sum(doc_id in gold_set for doc_id in retrieved)

            if gold_set:
                recall_sum += hit_count / len(gold_set)
            success_count += int(hit_count > 0)

            rel_rank = row["first_rel_rank"]
            if rel_rank is not None and rel_rank <= k:
                mrr_sum += 1.0 / float(rel_rank)

        denom = float(query_count) if query_count > 0 else 1.0
        metrics.append(
            {
                "dataset": bundle.name,
                "k": k,
                "recall_at_k": recall_sum / denom,
                "success_at_k": success_count / denom,
                "mrr_at_k": mrr_sum / denom,
                "num_queries": query_count,
            }
        )

    return {
        "dataset": bundle.name,
        "split_values": bundle.split_values,
        "query_count": len(bundle.queries),
        "doc_count": len(bundle.docs),
        "qrel_count": len(bundle.qrels),
        "queries_without_qrels": queries_without_qrels,
        "queries_without_relevant_docs": queries_without_relevant_docs,
        "metrics": metrics,
        "query_rows": query_rows,
    }

In [5]:
results_by_dataset: dict[str, dict[str, Any]] = {}

for dataset_name, paths in DATASETS.items():
    bundle = load_dataset(dataset_name, paths)
    results_by_dataset[dataset_name] = evaluate_dataset(
        bundle=bundle,
        k_values=K_VALUES,
        relevance_threshold=RELEVANCE_THRESHOLD,
    )

    overview_rows: list[dict[str, Any]] = []
    metric_rows: list[dict[str, Any]] = []
    sample_rows: list[dict[str, Any]] = []

    for dataset_name, result in results_by_dataset.items():
        overview_rows.append(
            {
                "dataset": dataset_name,
                "split": ",".join(result["split_values"]),
                "queries": result["query_count"],
                "docs": result["doc_count"],
                "qrels": result["qrel_count"],
                "queries_without_qrels": result["queries_without_qrels"],
                "queries_without_relevant_docs": result["queries_without_relevant_docs"],
            }
        )

        metric_rows.extend(result["metrics"])

        for row in result["query_rows"][:SAMPLE_QUERY_COUNT]:
            top_k = max(K_VALUES)
            top_docs = row["top_doc_ids"][:top_k]
            top_scores = row["top_doc_scores"][:top_k]
            sample_rows.append(
                {
                    "dataset": dataset_name,
                    "query_id": row["query_id"],
                    "query_text": row["query_text"],
                    "gold_doc_ids": " | ".join(row["gold_doc_ids"]),
                    "top_doc_ids": " | ".join(top_docs),
                    "top_doc_scores": " | ".join([f"{s:.4f}" for s in top_scores]),
                }
            )

    overview_df = pd.DataFrame(overview_rows).sort_values("dataset").reset_index(drop=True)
    metrics_df = pd.DataFrame(metric_rows).sort_values(["dataset", "k"]).reset_index(drop=True)
    samples_df = pd.DataFrame(sample_rows).reset_index(drop=True)

    payload = {
        "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "experiment": {
            "retriever": "BM25Okapi",
            "python_version": PYTHON_VERSION,
            "package": "rank_bm25",
            "package_version": RANK_BM25_VERSION,
            "dataset_names": sorted(list(DATASETS.keys())),
            "split": "normalized:local",
            "corpus": "normalized docs.normalized.jsonl",
            "preprocessing": "regex-tokenization + lowercase",
            "tokenizer_regex": TOKEN_PATTERN.pattern,
            "bm25_params": {"k1": BM25_K1, "b": BM25_B, "epsilon": BM25_EPSILON},
            "k_values": K_VALUES,
            "relevance_threshold": RELEVANCE_THRESHOLD,
            "seed": "not-applicable (BM25 deterministic without random seed)",
            "metrics": ["Recall@k", "Success@k", "MRR@k"],
        },
        "overview": overview_rows,
        "metrics": metric_rows,
        "sample_rankings": sample_rows,
    }

    json_path = BM25_OUTPUT_DIR / "bm25_results.json"
    csv_path = BM25_OUTPUT_DIR / "bm25_results.csv"

    with json_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=True, indent=2)

    metrics_df.to_csv(csv_path, index=False, encoding="utf-8")

    print(f"JSON geschrieben: {json_path}")
    print(f"CSV geschrieben: {csv_path}")

JSON geschrieben: <PROJEKT>/code/output/bm25/bm25_results.json
CSV geschrieben: <PROJEKT>/code/output/bm25/bm25_results.csv
JSON geschrieben: <PROJEKT>/code/output/bm25/bm25_results.json
CSV geschrieben: <PROJEKT>/code/output/bm25/bm25_results.csv


In [6]:
pd.set_option("display.max_colwidth", 180)

print("Datensatzübersicht")
display(overview_df)

print("Metrikresultate")
display(metrics_df)

print("Beispielrankings")
display(samples_df)

Datensatzübersicht


,dataset,split,queries,docs,qrels,queries_without_qrels,queries_without_relevant_docs
0,gerlerb,local,367,85738,367,0,0
1,germanquad,local,2204,474,2204,0,0


Metrikresultate


,dataset,k,recall_at_k,mrr_at_k,num_queries
0,gerlerb,1,0.217984,0.217984,367
1,gerlerb,5,0.359673,0.272025,367
2,gerlerb,10,0.411444,0.279218,367
3,germanquad,1,0.837568,0.837568,2204
4,germanquad,5,0.948276,0.882116,2204
5,germanquad,10,0.964610,0.884404,2204


Beispielrankings


,dataset,query_id,query_text,gold_doc_ids,top_doc_ids,top_doc_scores
0,germanquad,q40369,Was kann den Verschleiß des seillosen Aufzuges minimieren?,c40369,c40369 | c40389 | c38273 | c36674 | c38297 | c41062 | c37475 | c41277 | c41153 | c38332,27.2030 | 11.1640 | 8.7295 | 8.6020 | 8.5640 | 8.2233 | 8.0593 | 8.0233 | 7.9658 | 7.9412
1,germanquad,q40370,In welcher deutschen Stadt wird der seillose Aufzug getestet?,c40369,c40369 | c40436 | c42112 | c36764 | c41536 | c38590 | c41142 | c38320 | c36774 | c41033,28.5611 | 14.3085 | 13.7561 | 13.0344 | 11.8988 | 11.4067 | 10.9721 | 10.7456 | 10.6749 | 10.5301
2,germanquad,q40366,Wo wurde ein seilloser Aufzug entwickelt?,c40369,c40369 | c37644 | c37018 | c38247 | c37217 | c41119 | c36598 | c41161 | c40511 | c38194,27.4796 | 7.2742 | 6.0879 | 6.0210 | 5.9655 | 5.9005 | 5.7998 | 5.6947 | 5.5901 | 5.3998
3,germanquad,q40367,Wie funktioniert ein seilloser Aufzug?,c40369,c40369 | c36606 | c37099 | c44122 | c41536 | c36811 | c42127 | c41156 | c45642 | c38816,22.7762 | 3.3126 | 3.3058 | 3.2266 | 3.2122 | 3.1672 | 3.0305 | 3.0247 | 3.0190 | 2.9074
4,germanquad,q40368,Wann muss man die Zieletage in seillosen Aufzügen auswählen?,c40369,c40369 | c38189 | c40462 | c41445 | c38077 | c40219 | c37033 | c37277 | c41192 | c36974,16.0760 | 12.8726 | 12.4375 | 12.2665 | 10.5406 | 10.0996 | 9.8856 | 9.7670 | 9.4769 | 9.4324
5,gerlerb,206,Kann die Ablehnung der Annahme einer Verfassungsbeschwerde ohne Begründung erfolgen?,93d_bverfgg,93d_bverfgg | 93b_bverfgg | 93_bverfgg | 102_btgo_1980 | 93a_bverfgg | 32_bverfgg | 20_lwvfg | 2202_bgb | 145_sgg | 95_bverfgg,48.3466 | 36.7073 | 32.0422 | 30.5326 | 29.3310 | 29.0855 | 28.6887 | 28.5883 | 28.1260 | 27.8949
6,gerlerb,207,Wann muss der Arbeitgeber den Betriebsrat bei einer neuen Zuordnung von Mitarbeitern zu einer Entgeltgruppe informieren?,99_betrvg,14_entgtranspg | 15_entgtranspg | 85_betrvg | 80_betrvg | 96_betrvg | 43_betrvg | 89_betrvg | 92_betrvg | 12_asig | 90_betrvg,45.2781 | 42.1811 | 38.0834 | 37.5490 | 36.7376 | 36.6802 | 36.4852 | 36.2770 | 36.2583 | 35.7403
7,gerlerb,209,"Unter welchen Bedingungen kann der Zeitraum zur Berechnung des Elterngeldes verlängert werden, wenn es zu einer krankheitsbedingten Arbeitsunfähigkeit durch Schwangerschaft kommt?",2b_beeg,3_entgfg | 146_sgb_3 | 47_sgb_5 | 24b_sgb_5 | 574a_bgb | 10_gfdv | 26_sgb_10 | 162_kagb | 31_vwvfg | 35_tkg_2021,38.2758 | 35.8362 | 33.9728 | 33.8792 | 33.2348 | 32.0724 | 32.0300 | 31.9680 | 31.8253 | 31.3894
8,gerlerb,210,Kann man Monate mit schwangerschaftsbedingter Krankheit vom Bemessungszeitraum fürs Elterngeld ausnehmen?,2b_beeg,3_beeg | 150_sgb_3 | 21_sgb_6 | 9_endlagervlv | 2c_beeg | 150a_sgb_11 | 4_beeg | 2_beeg | 7_beeg | 4c_beeg,23.5658 | 20.8847 | 19.1047 | 16.9754 | 16.8101 | 16.7734 | 16.7029 | 16.4356 | 16.2108 | 16.1766
9,gerlerb,211,Bis zu welchem Alter eines Kindes kann ein Elternteil Elternzeit beanspruchen?,15_beeg,16_beeg | 15_beeg | 1_erzurlsoldv | 56_sgb_6 | 4_beeg | 11_bgb | 51_sgb_8 | 1629_bgb | 1680_bgb | 1612a_bgb,35.0554 | 32.1463 | 29.8943 | 27.9199 | 27.4494 | 27.1267 | 26.8905 | 26.8037 | 26.6826 | 26.4274


## Limitationen und Befunde

- Dieser PoC evaluiert reines BM25-Retrieval auf normalisierten Daten.
- `Recall@k` misst die Abdeckung der bekannten relevanten Dokumente; `Success@k` misst, ob mindestens ein bekanntes relevantes Dokument in Top-k erscheint.
- Bei den aktuell lokal vorliegenden Single-Relevance-qrels sind `Recall@k` und `Success@k` numerisch gleich. Das belegt keine vollständige Relevanzmenge und begrenzt Aussagen über alternative relevante Treffer.
- Queries ohne qrels oder ohne relevante Dokumente im Corpus werden mit 0 in den Metriken gewertet; ihre Anzahl wird in der Übersicht ausgewiesen.
- Ergebnisse und Beispielrankings sind in den Tabellen sichtbar und als strukturierte Dateien unter `code/output/bm25/` gespeichert.